# Lattice Regularization of Topological Solitons: 3D Framing, the Mass Hierarchy, and the WIMP Miracle
## Interactive Model Demonstration

This notebook demonstrates the **Level I (Computational)** aspect of the EQIT framework. It shows how pure topological invariants of mathematical knots are converted into physical mass predictions through a discrete Hamiltonian.

We use Google Colab to provide a fully transparent, self-contained, and reproducible environment for the reviewers.

In [ ]:
import torch
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

print("Libraries loaded successfully!")

Libraries loaded successfully!


### Step 1: Algebraic Knot Invariants
To ensure no parameters are fit ad hoc, the topological penalty is strictly derived from the Alexander polynomial $\Delta(t)$ evaluated at the fermionic limit ($t = -1$). This value is known as the knot determinant $C$.

In [ ]:
t = sp.Symbol('t')

# Alexander polynomials for the fundamental knot sequence used in EQIT
alexander_polynomials = {
    '3_1 (Electron)': t - 1 + t**(-1),
    '4_1 (Muon)': -t + 3 - t**(-1),
    '5_2 (Tau)': 2*t - 3 + 2*t**(-1),
    '8_3 (Dark Matter)': t**2 - 4*t + 7 - 4*t**(-1) + t**(-2)
}

knot_db = {}
print("Rigorous Knot Determinants C = |Δ(-1)|:")
print("-" * 40)
for name, poly in alexander_polynomials.items():
    det = abs(poly.subs(t, -1))
    crossings = int(name.split('_')[0][0])
    knot_db[name] = {'K': crossings, 'C': float(det)}
    print(f"{name:20s} | K={crossings} | C={det}")

Rigorous Knot Determinants C = |Δ(-1)|:
----------------------------------------
3_1 (Electron)       | K=3 | C=3
4_1 (Muon)           | K=4 | C=5
5_2 (Tau)            | K=5 | C=7
8_3 (Dark Matter)    | K=8 | C=17


### Step 2: The Discrete Topological Hamiltonian
We construct a $2^K \times 2^K$ Hermitian matrix representing the phase space of the knot crossings. The diagonal elements represent the geometric tension and topological rigidity ($C$), while the off-diagonal elements represent tunneling (Reidemeister transitions).

In [ ]:
class TopologicalHamiltonian:
    def __init__(self, mu, beta, lam, delta, B):
        self.mu = mu
        self.beta = beta
        self.lam = lam
        self.delta = delta
        self.B = B

    def build_matrix(self, K, C):
        N = 2**K
        H = torch.zeros((N, N), dtype=torch.complex128)

        # 1. Diagonal: Geometric tension and Topological penalty
        C_tensor = torch.ones(N, dtype=torch.float64)
        C_tensor[0] = C   # Fully untwisted mirror state
        C_tensor[-1] = C  # Fully twisted mirror state

        tension = self.mu * torch.exp(torch.tensor(self.beta * float(K)))
        diag_energies = (tension * float(K) - self.lam * C_tensor).to(torch.complex128)
        diag_indices = torch.arange(N)
        H[diag_indices, diag_indices] = diag_energies

        # 2. Off-diagonal elements (Hypercube tunneling)
        i = torch.arange(N).view(N, 1)
        j = torch.arange(N).view(1, N)
        xor_matrix = torch.bitwise_xor(i, j)
        is_power_of_2 = (torch.bitwise_and(xor_matrix, xor_matrix - 1) == 0) & (xor_matrix != 0)
        H[is_power_of_2] = -self.delta / float(K)

        # 3. Parity violation coupling between absolute mirror states
        for idx in range(N // 2):
            mirror_idx = (N - 1) - idx
            H[idx, mirror_idx] = 1j * self.B
            H[mirror_idx, idx] = -1j * self.B

        return H

    def get_ground_state_energy(self, K, C):
        H = self.build_matrix(K, C)
        eigenvalues = torch.linalg.eigvalsh(H)
        return eigenvalues[0].item() # Lowest energy state

print("Hamiltonian class compiled successfully.")

Hamiltonian class compiled successfully.


### Step 3: Spectral Calibration & Blind Prediction
We use the optimal parameters found via the Inverse Spectral Problem (calibrated exclusively on the electron and muon masses). We then blindly predict the $\tau$-lepton mass.

In [ ]:
# Optimal parameters calibrated on e and μ
mu_opt = 0.002588
lam_opt = 4.108225
beta_opt = 2.5
delta_opt = 0.020009
B_opt = 0.715517

model = TopologicalHamiltonian(mu=mu_opt, beta=beta_opt, lam=lam_opt, delta=delta_opt, B=B_opt)

# Calculate Ground State Energies
E_e = model.get_ground_state_energy(knot_db['3_1 (Electron)']['K'], knot_db['3_1 (Electron)']['C'])
E_mu = model.get_ground_state_energy(knot_db['4_1 (Muon)']['K'], knot_db['4_1 (Muon)']['C'])
E_tau = model.get_ground_state_energy(knot_db['5_2 (Tau)']['K'], knot_db['5_2 (Tau)']['C'])

# Experimental masses in MeV (Particle Data Group)
m_e_exp = 0.510998
m_mu_exp = 105.658
m_tau_exp = 1776.86

# Map energies to masses
m_e_calc = E_e / E_e * m_e_exp
m_mu_calc = E_mu / E_e * m_e_exp
m_tau_calc = E_tau / E_e * m_e_exp

print("=== EQIT Mass Predictions ===")
print(f"Electron (3_1): {m_e_calc:.3f} MeV (Calibration anchor)")
print(f"Muon (4_1)    : {m_mu_calc:.3f} MeV (Calibration anchor)")
print(f"Tau (5_2)     : {m_tau_calc:.3f} MeV (BLIND PREDICTION)")
print("-" * 30)
error_tau = abs(m_tau_calc - m_tau_exp) / m_tau_exp * 100
print(f"Experimental Tau Mass: {m_tau_exp:.3f} MeV")
print(f"Prediction Error: {error_tau:.2f}%")

=== EQIT Mass Predictions ===
Electron (3_1): 0.511 MeV (Calibration anchor)
Muon (4_1)    : 105.926 MeV (Calibration anchor)
Tau (5_2)     : 1763.795 MeV (BLIND PREDICTION)
------------------------------
Experimental Tau Mass: 1776.860 MeV
Prediction Error: 0.74%


### Step 4: Dark Matter Candidate Prediction
Extrapolating the calibrated vacuum parameters to the electromagnetically neutral $8_3$ amphichiral knot.

In [ ]:
E_dm = model.get_ground_state_energy(knot_db['8_3 (Dark Matter)']['K'], knot_db['8_3 (Dark Matter)']['C'])
m_dm_calc = E_dm / E_e * m_e_exp
m_dm_TeV = m_dm_calc / 1e6

print(f"Predicted Mass for 8_3 Dark Matter Candidate: {m_dm_TeV:.3f} TeV")

Predicted Mass for 8_3 Dark Matter Candidate: 5.146 TeV
